In [80]:
import os

print(os.getcwd())

d:\NuralRetail\notebooks


In [81]:
# =========================================================
# 📦 Imports
# =========================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from sklearn.preprocessing import MinMaxScaler

In [82]:
# =========================================================
# 📂 Load Dataset
# =========================================================

sales_df = pd.read_csv("../data/processed/cleaned_retail_data.csv")

sales_df.head()

,invoiceno,stockcode,description,quantity,invoicedate,unitprice,customerid,country,totalprice,total_amount,year,month,day,day_of_week,weekofyear,hour,year_month,log_quantity,log_total_amount
0,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0,30.0,2009,12,1,Tuesday,49,7,2009-12,3.218876,3.433987
1,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085.0,United Kingdom,39.6,39.6,2009,12,1,Tuesday,49,7,2009-12,3.218876,3.703768
2,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0,30.0,2009,12,1,Tuesday,49,7,2009-12,3.218876,3.433987
3,489435,22350,CAT BOWL,12,2009-12-01 07:46:00,2.55,13085.0,United Kingdom,30.6,30.6,2009,12,1,Tuesday,49,7,2009-12,2.564949,3.453157
4,489435,22195,HEART MEASURING SPOONS LARGE,24,2009-12-01 07:46:00,1.65,13085.0,United Kingdom,39.6,39.6,2009,12,1,Tuesday,49,7,2009-12,3.218876,3.703768


In [83]:
# =========================================================
# 🧹 Basic Cleaning
# =========================================================

sales_df.dropna(inplace=True)

sales_df = sales_df[sales_df["quantity"] > 0]

sales_df = sales_df[sales_df["unitprice"] > 0]

sales_df["invoicedate"] = pd.to_datetime(sales_df["invoicedate"])

sales_df.head()

,invoiceno,stockcode,description,quantity,invoicedate,unitprice,customerid,country,totalprice,total_amount,year,month,day,day_of_week,weekofyear,hour,year_month,log_quantity,log_total_amount
0,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0,30.0,2009,12,1,Tuesday,49,7,2009-12,3.218876,3.433987
1,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085.0,United Kingdom,39.6,39.6,2009,12,1,Tuesday,49,7,2009-12,3.218876,3.703768
2,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0,30.0,2009,12,1,Tuesday,49,7,2009-12,3.218876,3.433987
3,489435,22350,CAT BOWL,12,2009-12-01 07:46:00,2.55,13085.0,United Kingdom,30.6,30.6,2009,12,1,Tuesday,49,7,2009-12,2.564949,3.453157
4,489435,22195,HEART MEASURING SPOONS LARGE,24,2009-12-01 07:46:00,1.65,13085.0,United Kingdom,39.6,39.6,2009,12,1,Tuesday,49,7,2009-12,3.218876,3.703768


In [84]:
# =========================================================
# 📦 Inventory Aggregation
# =========================================================

sales_df["revenue"] = sales_df["quantity"] * sales_df["unitprice"]

inventory_df = sales_df.groupby('stockcode').agg({

    'quantity': 'sum',
    'total_amount': 'sum',
    'invoiceno': 'nunique',
    'unitprice': 'mean'

}).reset_index()

inventory_df.columns = [

    'product',
    'annual_demand',
    'annual_revenue',
    'order_frequency',
    'unit_price'
]

inventory_df.head()


,product,annual_demand,annual_revenue,order_frequency,unit_price
0,10002,3225,2724.87,270,0.847636
1,10080,303,124.61,26,0.509259
2,10109,4,1.68,1,0.420000
3,10120,648,136.08,62,0.210000
4,10123C,532,175.88,45,0.623778


## DEMAND ENGINEERING

In [85]:
inventory_df['daily_demand'] = (
    inventory_df['annual_demand'] / 365
)

inventory_df['ordering_cost'] = np.random.randint(
    50,
    300,
    len(inventory_df)
)

inventory_df['holding_cost'] = (
    inventory_df['unit_price'] * 0.15
)

inventory_df['lead_time_days'] = np.random.randint(
    3,
    15,
    len(inventory_df)
)

inventory_df['current_stock'] = np.random.randint(
    20,
    500,
    len(inventory_df)
)

inventory_df['monthly_sales'] = (
    inventory_df['annual_demand'] / 12
)

inventory_df['demand_std'] = np.random.uniform(
    5,
    40,
    len(inventory_df)
)

inventory_df.head()

,product,annual_demand,annual_revenue,order_frequency,unit_price,daily_demand,ordering_cost,holding_cost,lead_time_days,current_stock,monthly_sales,demand_std
0,10002,3225,2724.87,270,0.847636,8.835616,266,0.127145,8,194,268.750000,12.810364
1,10080,303,124.61,26,0.509259,0.830137,73,0.076389,14,290,25.250000,12.379918
2,10109,4,1.68,1,0.420000,0.010959,135,0.063000,7,82,0.333333,24.558029
3,10120,648,136.08,62,0.210000,1.775342,254,0.031500,11,303,54.000000,38.242549
4,10123C,532,175.88,45,0.623778,1.457534,194,0.093567,11,423,44.333333,26.304155


## EOQ MODEL

In [86]:
inventory_df['EOQ'] = np.sqrt(

    (
        2
        * inventory_df['annual_demand']
        * inventory_df['ordering_cost']
    )

    /

    (inventory_df['holding_cost'] + 1)
)

inventory_df[['product', 'EOQ']].head()

,product,EOQ
0,10002,1233.760022
1,10080,202.727721
2,10109,31.874637
3,10120,564.917129
4,10123C,434.459249


## SAFETY STOCK

In [87]:
inventory_df['Safety_Stock'] = (

    inventory_df['daily_demand']

    *

    inventory_df['lead_time_days']

    * 0.5
)

inventory_df[['product', 'Safety_Stock']].head()

,product,Safety_Stock
0,10002,35.342466
1,10080,5.810959
2,10109,0.038356
3,10120,9.764384
4,10123C,8.016438


## REORDER POINT ENGINE

In [88]:
inventory_df['Reorder_Point'] = (

    inventory_df['daily_demand']

    *

    inventory_df['lead_time_days']

    +

    inventory_df['Safety_Stock']
)

inventory_df[['product', 'Reorder_Point']].head()

,product,Reorder_Point
0,10002,106.027397
1,10080,17.432877
2,10109,0.115068
3,10120,29.293151
4,10123C,24.049315


## REORDER ALERT ENGINE

In [89]:
inventory_df['Reorder_Alert'] = (

    inventory_df['current_stock']

    <

    inventory_df['Reorder_Point']
)

inventory_df['Reorder_Alert'].value_counts()

Reorder_Alert
False    4317
True      268
Name: count, dtype: int64

##  INVENTORY CLASSIFICATION (ABC)

In [90]:
inventory_df['Revenue_Share'] = (

    inventory_df['annual_revenue']

    /

    inventory_df['annual_revenue'].sum()
)

inventory_df = inventory_df.sort_values(
    by='Revenue_Share',
    ascending=False
)

inventory_df['Cumulative_Revenue'] = (
    inventory_df['Revenue_Share'].cumsum()
)

inventory_df['ABC_Class'] = np.where(

    inventory_df['Cumulative_Revenue'] <= 0.80,

    'A',

    np.where(

        inventory_df['Cumulative_Revenue'] <= 0.95,

        'B',

        'C'
    )
)

inventory_df[['product', 'ABC_Class']].head()

,product,ABC_Class
4036,85123A,A
4014,85099B,A
1599,22423,A
3786,84879,A
244,20725,A


## XYZ CLASSIFICATION

In [91]:
inventory_df['CV'] = (

    inventory_df['demand_std']

    /

    (inventory_df['daily_demand'] + 1)
)

inventory_df['XYZ_Class'] = np.where(

    inventory_df['CV'] < 0.5,

    'X',

    np.where(

        inventory_df['CV'] < 1.0,

        'Y',

        'Z'
    )
)

inventory_df[['product', 'XYZ_Class']].head()

,product,XYZ_Class
4036,85123A,X
4014,85099B,X
1599,22423,Z
3786,84879,X
244,20725,X


## INVENTORY RISK ENGINE

In [92]:
inventory_df['Stock_Coverage_Days'] = (

    inventory_df['current_stock']

    /

    (inventory_df['daily_demand'] + 1)
)

inventory_df['Inventory_Risk_Score'] = np.where(

    inventory_df['Stock_Coverage_Days'] < 7,

    90,

    np.where(

        inventory_df['Stock_Coverage_Days'] < 15,

        70,

        np.where(

            inventory_df['Stock_Coverage_Days'] < 30,

            40,

            10
        )
    )
)

inventory_df[['product', 'Inventory_Risk_Score']].head()

,product,Inventory_Risk_Score
4036,85123A,90
4014,85099B,90
1599,22423,10
3786,84879,90
244,20725,70


## DEAD STOCK DETECTION

In [93]:
inventory_df['Dead_Stock'] = np.where(

    inventory_df['monthly_sales'] < 5,

    1,

    0
)

inventory_df['Dead_Stock'].value_counts()

Dead_Stock
0    3532
1    1053
Name: count, dtype: int64

##  KPI SUMMARY

In [94]:
print('📦 INVENTORY KPI SUMMARY')
print('-' * 50)

print(
    'Total Products:',
    len(inventory_df)
)

print(
    'Reorder Alerts:',
    inventory_df['Reorder_Alert'].sum()
)

print(
    'Average Risk Score:',
    round(
        inventory_df['Inventory_Risk_Score'].mean(),
        2
    )
)

print(
    'Dead Stock Products:',
    inventory_df['Dead_Stock'].sum()
)

📦 INVENTORY KPI SUMMARY
--------------------------------------------------
Total Products: 4585
Reorder Alerts: 268
Average Risk Score: 19.15
Dead Stock Products: 1053


## INVENTORY RISK DISTRIBUTION

In [95]:
fig = px.histogram(

    inventory_df,

    x='Inventory_Risk_Score',

    nbins=20,

    title='Inventory Risk Distribution'
)

fig.show()

## TOP HIGH-RISK PRODUCTS

In [96]:
top_inventory = inventory_df.sort_values(

    by='Inventory_Risk_Score',

    ascending=False

).head(20)

fig = px.bar(

    top_inventory,

    x='product',

    y=[
        'current_stock',
        'Reorder_Point'
    ],

    barmode='group',

    title='Top 20 High-Risk Products'
)

fig.show()

##  ABC ANALYSIS

In [97]:
abc_counts = inventory_df['ABC_Class'].value_counts()

fig = px.pie(

    values=abc_counts.values,

    names=abc_counts.index,

    title='ABC Inventory Classification'
)

fig.show()

##  XYZ ANALYSIS

In [98]:
xyz_counts = inventory_df['XYZ_Class'].value_counts()

fig = px.bar(

    x=xyz_counts.index,

    y=xyz_counts.values,

    title='XYZ Demand Classification'
)

fig.show()

## DEAD STOCK ANALYSIS

In [99]:
dead_stock = inventory_df[

    inventory_df['Dead_Stock'] == 1
]

print(
    'Dead Stock Products:',
    len(dead_stock)
)

dead_stock.head()

Dead Stock Products: 1053


,product,annual_demand,annual_revenue,order_frequency,unit_price,daily_demand,ordering_cost,holding_cost,lead_time_days,current_stock,...,Reorder_Point,Reorder_Alert,Revenue_Share,Cumulative_Revenue,ABC_Class,CV,XYZ_Class,Stock_Coverage_Days,Inventory_Risk_Score,Dead_Stock
1981,22824,41,1470.95,41,35.876829,0.112329,248,5.381524,4,311,...,0.673973,False,0.000170,0.847330,B,20.535671,Z,279.593596,10,1
2600,23485,53,1325.00,53,25.000000,0.145205,257,3.750000,3,91,...,0.653425,False,0.000153,0.863861,B,27.452143,Z,79.461722,10,1
2577,23462,52,1037.40,39,19.950000,0.142466,182,2.992500,8,251,...,1.709589,False,0.000120,0.895944,B,30.204624,Z,219.700240,10,1
1960,22802,52,953.40,39,19.026923,0.142466,64,2.854038,8,222,...,1.709589,False,0.000110,0.907920,B,16.004752,Z,194.316547,10,1
2389,23253,58,925.10,33,15.950000,0.158904,133,2.392500,9,66,...,2.145205,False,0.000107,0.910418,B,29.576798,Z,56.950355,10,1


##  EXPORT DASHBOARD DATA

In [100]:
inventory_export = inventory_df[[

    'product',
    'annual_demand',
    'annual_revenue',
    'order_frequency',
    'unit_price',
    'daily_demand',
    'ordering_cost',
    'holding_cost',
    'lead_time_days',
    'current_stock',
    'monthly_sales',
    'demand_std',
    'EOQ',
    'Safety_Stock',
    'Reorder_Point',
    'Reorder_Alert',
    'CV',
    'XYZ_Class',
    'Inventory_Risk_Score',
    'Dead_Stock'
]]

inventory_export.to_csv(

    '../app/dashboard/data/inventory_risk.csv',

    index=False
)

print('✅ inventory_risk.csv exported successfully')

✅ inventory_risk.csv exported successfully


## FINAL VALIDATION

In [101]:
check_df = pd.read_csv(

    '../app/dashboard/data/inventory_risk.csv'
)

print(check_df.head())

print(check_df.columns)

  product  annual_demand  annual_revenue  order_frequency  unit_price  \
0  85123A          22782        66919.40             3681    2.947526   
1  85099B          24155        48247.80             2640    2.001975   
2   22423           3494        44534.26             2020   12.781810   
3   84879          26243        44302.67             2134    1.688232   
4   20725          19743        32585.55             2344    1.657585   

   daily_demand  ordering_cost  holding_cost  lead_time_days  current_stock  \
0     62.416438            290      0.442129              10            138   
1     66.178082            210      0.300296              12            461   
2      9.572603            146      1.917271               7            341   
3     71.898630            216      0.253235               5            288   
4     54.090411            176      0.248638               8            440   

   monthly_sales  demand_std          EOQ  Safety_Stock  Reorder_Point  \
0    1898.50